# Notebook 01: Data Loading and SQL Workflow

### Purpose
- Validates the integrity of a three-arm randomized email marketing experiment before any outcome analysis.
- Rigorous data validation is the foundation of trustworthy experimentation — this notebook establishes that foundation.

### Objectives
- Load 64,000 customer records into DuckDB and verify schema, arm assignments, and data completeness.
- Test for Sample Ratio Mismatch (SRM) — an imbalanced arm is a signal of broken randomization, not a real effect.
- Compute baseline visit, conversion, and spend rates per arm to orient the analysis.
- Confirm covariate balance across arms as evidence that randomization succeeded.
- Characterize the spend distribution: heavily zero-inflated with a long right tail, motivating winsorization in later analysis.

## 1. Get, load, and verify dataset

In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import urllib.request

data_path = Path("../data/raw/hillstrom.csv")

if not data_path.exists():
    url = "http://www.minethatdata.com/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, data_path)
    print("Downloaded.")
else:
    print(f"Data already exists ({data_path.stat().st_size / 1024:.0f} KB).")

Data already exists (3872 KB).


In [11]:
hillstrom_df = pd.read_csv(data_path)

display(hillstrom_df.head(), hillstrom_df.tail())

display(hillstrom_df.describe())

display(hillstrom_df.info())


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


,recency,history,mens,womens,newbie,visit,conversion,spend
count,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,0.551031,0.549719,0.502250,0.146781,0.009031,1.050908
std,3.507592,256.158608,0.497393,0.497526,0.499999,0.353890,0.094604,15.036448
min,1.000000,29.990000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,64.660000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.000000,158.110000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,9.000000,325.657500,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,12.000000,3345.930000,1.000000,1.000000,1.000000,1.000000,1.000000,499.000000


<class 'pandas.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  str    
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  str    
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  str    
 8   segment          64000 non-null  str    
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), str(4)
memory usage: 5.9 MB


None

In [17]:
print(f"Duplicate rows: {hillstrom_df.duplicated().sum()}")

Duplicate rows: 6562


In [ ]:
for col in hillstrom_df.select_dtypes(include="str").columns:
    print(f"{col}\n{hillstrom_df[col].value_counts()}\n")